# Clase 013 — Type hints y mypy

**Parte 0** · Ramalho cap. 8 + PEP 484.

> 🎯 Tipos como documentación verificable. mypy detecta bugs antes de runtime.

> ⏱️ ~75 min

## ⚙️ Setup

In [ ]:
from typing import Optional, Literal, TypedDict, Protocol, TypeAlias
from dataclasses import dataclass

## 1️⃣ Sintaxis básica

```python
def saludar(nombre: str, formal: bool = False) -> str:
    return f'Buenos días, {nombre}' if formal else f'Hola {nombre}'
```

**Importante**: los tipos son **anotaciones** — Python NO los verifica en runtime. Son para tooling (IDE, mypy, IA).

In [ ]:
def saludar(nombre: str, formal: bool = False) -> str:
    return f'Buenos días, {nombre}' if formal else f'Hola {nombre}'

print(saludar('Ana'))
print(saludar('Bob', formal=True))

# Esto NO falla en runtime (Python no verifica), pero mypy lo detectaría:
print(saludar(123))  # type hint dice str, le pasamos int

## 2️⃣ Tipos compuestos modernos

Desde Python 3.9+, usa **lowercase** built-ins:

```python
# ✅ moderno (3.9+)
def f(xs: list[int], lookup: dict[str, float]) -> tuple[int, str]:
    ...

# ❌ viejo (pre-3.9)
from typing import List, Dict, Tuple
def f(xs: List[int], lookup: Dict[str, float]) -> Tuple[int, str]:
    ...
```

Desde 3.10+, usa `|` para uniones:

```python
# ✅ moderno (3.10+)
def parse(x: str | int) -> float | None:
    ...

# ❌ viejo
from typing import Union, Optional
def parse(x: Union[str, int]) -> Optional[float]:
    ...
```

In [ ]:
# Demo: tipos compuestos
def promedios_por_grupo(
    datos: list[dict[str, float]],
    grupo_key: str = 'grupo',
    valor_key: str = 'valor',
) -> dict[str, float]:
    sumas: dict[str, float] = {}
    conteos: dict[str, int] = {}
    for d in datos:
        g = d[grupo_key]
        sumas[g] = sumas.get(g, 0.0) + d[valor_key]
        conteos[g] = conteos.get(g, 0) + 1
    return {g: sumas[g] / conteos[g] for g in sumas}

datos = [
    {'grupo': 'A', 'valor': 10.0},
    {'grupo': 'A', 'valor': 20.0},
    {'grupo': 'B', 'valor': 5.0},
]
print(promedios_por_grupo(datos))

## 3️⃣ `Optional` vs default

Distinción importante:

```python
def f(x: int = 0):           # x es int; default 0 si no se pasa
def f(x: int | None = None): # x puede ser None — el caller debe decidir
```

El segundo caso obliga al cuerpo a manejar `None`:

In [ ]:
def buscar(nombre: str, default: int | None = None) -> int:
    db = {'Ana': 30, 'Bob': 25}
    if nombre in db:
        return db[nombre]
    if default is None:
        raise KeyError(nombre)
    return default

print(buscar('Ana'))
print(buscar('Cris', default=0))
try:
    buscar('Cris')
except KeyError as e:
    print(f'KeyError: {e}')

## 4️⃣ `TypedDict` — diccionarios con esquema

Útil cuando recibes JSON o configs:

In [ ]:
class PersonaDict(TypedDict):
    nombre: str
    edad: int
    activo: bool

def saludar_persona(p: PersonaDict) -> str:
    return f'{p["nombre"]} ({p["edad"]}) está {"activo" if p["activo"] else "inactivo"}'

p: PersonaDict = {'nombre': 'Ana', 'edad': 30, 'activo': True}
print(saludar_persona(p))

## 5️⃣ `Literal` — valores concretos como tipo

Útil para parámetros que solo aceptan ciertos strings:

In [ ]:
def ordenar(items: list[int], orden: Literal['asc', 'desc'] = 'asc') -> list[int]:
    return sorted(items, reverse=(orden == 'desc'))

print(ordenar([3, 1, 4, 1, 5]))
print(ordenar([3, 1, 4, 1, 5], orden='desc'))
# mypy detectaría: ordenar([1,2], orden='upward')  # 'upward' no es 'asc'|'desc'

## 6️⃣ `Protocol` — duck typing tipado

"Cualquier cosa que tenga estos métodos":

In [ ]:
class TienePromedio(Protocol):
    def promedio(self) -> float: ...

@dataclass
class Curso:
    notas: list[float]
    def promedio(self) -> float:
        return sum(self.notas) / len(self.notas)

@dataclass
class Atleta:
    tiempos: list[float]
    def promedio(self) -> float:
        return sum(self.tiempos) / len(self.tiempos)

def reportar(items: list[TienePromedio]) -> None:
    for it in items:
        print(f'{type(it).__name__}: {it.promedio():.2f}')

reportar([Curso([6.5, 7.0]), Atleta([10.1, 9.8])])

## 7️⃣ Correr mypy

```bash
pip install mypy
mypy archivo.py            # modo permisivo
mypy --strict archivo.py   # modo estricto (recomendado para libs)
```

Config en `pyproject.toml`:

```toml
[tool.mypy]
python_version = "3.12"
strict = true
ignore_missing_imports = true   # libs sin stubs
```

**`# type: ignore`** al final de una línea silencia mypy en esa línea — escape hatch para casos legítimos (libs sin stubs, hacks intencionales).

## 8️⃣ ¿Cuándo SÍ, cuándo NO?

**Sí**:
- APIs públicas (funciones que importan otros)
- Data classes / records
- Lógica de dominio compleja
- Tipos que mejoran autocompletado

**Quizá no**:
- Notebooks puramente exploratorios
- Scripts one-shot
- Cuando el tipo es obvio y agregar ruido (`x = 5  # int` no aporta)

## ✅ Checklist

- [ ] Anoto funciones públicas con tipos en params y retorno
- [ ] Uso `list[int]` (3.9+) en vez de `List[int]`
- [ ] Uso `X | None` (3.10+) en vez de `Optional[X]`
- [ ] Sé correr mypy y leer sus errores
- [ ] Conozco `TypedDict`, `Literal`, `Protocol`

## 📝 Homework

Ver `README.md`. Módulo `analytics.py` con 5+ funciones anotadas, `pyproject.toml` con `[tool.mypy] strict=true`, log de mypy sin errores.

## 📖 Definiciones y características

**Type hint (annotation)**

Anotación de tipo en signature o variable: `def f(x: int) -> str`. **Python NO verifica en runtime** — son metadata leída por IDE/linter/mypy. Disponibles en `f.__annotations__`.

**`Optional[X]` / `X | None`**

Indica que el valor puede ser `X` o `None`. `Optional[X]` ≡ `Union[X, None]` ≡ `X | None` (PEP 604, 3.10+). Usa la sintaxis con `|` en código nuevo.

**`TypedDict`**

Esquema para diccionarios: `class PersonaDict(TypedDict): nombre: str; edad: int`. Permite tipear dicts que vienen de JSON/API sin convertirlos a dataclass.

**`Literal`**

Restringe a un conjunto de valores: `Literal['asc', 'desc']` solo acepta esas 2 strings. Útil para flags, modos, enums simples.

**`Protocol` (structural typing)**

Define interfaz por **estructura** (duck typing tipado): `class TienePromedio(Protocol): def promedio(self) -> float: ...`. Cualquier clase con `.promedio()` la satisface, sin heredarla.

**mypy**

Static type checker oficial. Lee tu código, sigue las anotaciones, reporta inconsistencias antes de ejecutar. Modo `--strict` lo hace exigente; recomendado para librerías.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `from typing import List` da deprecation warning en mypy | Python 3.9+ usa lowercase: `list[int]` en vez de `List[int]`. **Fix**: actualiza imports + sintaxis. |
| `Optional[int] = 0` confunde | `Optional[int]` admite `None`. Si tu default es `0`, no necesitas Optional. **Fix**: `x: int = 0` (no admite None) vs `x: int | None = None` (admite None). |
| mypy se queja "Cannot find module 'libreria'" | Lib sin type stubs publicados. **Fix**: `pip install types-libreria` si existe (PEP 561), o `[tool.mypy] ignore_missing_imports = true` en pyproject. |
| Anoté pero mypy no encuentra errores | mypy no se llamó. **Fix**: `mypy archivo.py`. No es automático — debe estar en CI o pre-commit. |
| `reveal_type(x)` no existe en runtime | Es exclusivo de mypy: lo lees como output del check, NO ejecutas. Si lo dejas en código que corre, lanza NameError. |

## ❓ Preguntas frecuentes

**❓ ¿Tipo hints son obligatorios?**

**No** — Python no los verifica. Pero son fuertemente recomendados en: APIs públicas (funciones que importan otros), librerías, código compartido. En notebooks exploratorios, casi nunca aportan.

**❓ ¿Slow my code los type hints?**

**No** — son metadata, no impactan runtime. `from __future__ import annotations` además las hace lazy (strings, evaluadas solo si introspeccionas).

**❓ ¿mypy en CI: estricto o permisivo?**

Empieza permisivo (sin `--strict`), arregla lo obvio, luego activa `--strict` gradualmente. Si arrancas estricto en un repo viejo, te ahogas en errores y termina ignorado.

**❓ ¿Y si no sé qué tipo poner?**

`Any` (de `typing`) es válido — desactiva el check para ese caso. Mejor que mentir con un tipo incorrecto. Convención: comentario `# TODO: tipo correcto` para revisión futura.

**❓ ¿Pydantic vs dataclass + type hints?**

**dataclass + hints**: tipos solo a nivel mypy, no en runtime. **Pydantic**: valida en runtime (parsing de JSON con coerción y errores legibles). Para input externo (API, config) → Pydantic. Para internal record → dataclass.

## 🔗 Referencias

- Ramalho, *Fluent Python* 2e, cap. 8
- [typing docs](https://docs.python.org/3/library/typing.html)
- [mypy docs](https://mypy.readthedocs.io/)

➡️ **Siguiente:** [014 — NumPy: tipos, creación, atributos](../014-numpy-tipos-creacion-atributos/README.md)